# Modelaci?n de Frecuencia - Responsabilidad Civil (RC)

Este notebook desarrolla modelos de regresi?n para predecir la frecuencia de siniestros en la cobertura de Responsabilidad Civil.

**Dataset**: `rc_siniestrados.csv` (67 registros)  
**Variable objetivo**: `Resp_Civil_siniestros_num`  
**Caracter?stica especial**: Todos los valores son 1.0 (modelo constante ?ptimo)

In [ ]:
# Importaci?n de librer?as
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configuraci?n de visualizaci?n
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Librer?as importadas correctamente")

In [ ]:
# Carga de datos
df = pd.read_csv('../data/processed/rc_siniestrados.csv')
print(f"Dataset cargado: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

In [ ]:
# An?lisis exploratorio de la variable objetivo
target_var = 'Resp_Civil_siniestros_num'
print(f"An?lisis de la variable objetivo: {target_var}")
print(f"Valores ?nicos: {df[target_var].unique()}")
print(f"Distribuci?n de frecuencias:")
print(df[target_var].value_counts().sort_index())

# Estad?sticas descriptivas
print(f"\nEstad?sticas descriptivas:")
print(df[target_var].describe())

In [ ]:
# Visualizaci?n de la distribuci?n de frecuencias
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
df[target_var].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Distribuci?n de Frecuencias RC')
plt.xlabel('Frecuencia de Siniestros')
plt.ylabel('Cantidad de Registros')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
plt.pie(df[target_var].value_counts().sort_index(), 
        labels=df[target_var].value_counts().sort_index().index,
        autopct='%1.1f%%', startangle=90)
plt.title('Proporci?n de Frecuencias RC')

plt.tight_layout()
plt.show()

print("\n=== OBSERVACI?N CR?TICA ===")
print("Todos los registros tienen frecuencia = 1.0")
print("El modelo ?ptimo es una CONSTANTE que predice siempre 1.0")
print("No hay variabilidad en la variable objetivo")

In [ ]:
# Preparaci?n de variables predictoras
# Selecci?n de variables independientes (excluyendo la variable objetivo y otras coberturas)
exclude_cols = ['Resp_Civil_siniestros_num', 'Resp_Civil_siniestros_monto',
                'Contenidos_siniestros_num', 'Contenidos_siniestros_monto',
                'Gastos_Adicionales_siniestros_num', 'Gastos_Adicionales_siniestros_monto',
                'Gastos_Medicos_RC_siniestros_num', 'Gastos_Medicos_RC_siniestros_monto']

feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Variables predictoras seleccionadas: {feature_cols}")

X = df[feature_cols].copy()
y = df[target_var].copy()

print(f"\nShape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"\nTipos de datos en X:")
print(X.dtypes)

In [ ]:
# Codificaci?n de variables categ?ricas
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

print(f"Variables categ?ricas: {categorical_cols}")
print(f"Variables num?ricas: {numerical_cols}")

# One-hot encoding para variables categ?ricas
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=False)
print(f"\nShape despu?s de encoding: {X_encoded.shape}")
print(f"Nuevas columnas: {list(X_encoded.columns)}")

In [ ]:
# Estandarizaci?n de variables num?ricas
scaler = StandardScaler()
X_scaled = X_encoded.copy()

if numerical_cols:
    # Identificar las columnas num?ricas en el dataset codificado
    numerical_cols_encoded = [col for col in X_encoded.columns if any(num_col in col for num_col in numerical_cols)]
    X_scaled[numerical_cols_encoded] = scaler.fit_transform(X_encoded[numerical_cols_encoded])
    print(f"Variables num?ricas estandarizadas: {numerical_cols_encoded}")
else:
    print("No hay variables num?ricas para estandarizar")

print(f"\nDataset final preparado: {X_scaled.shape}")
X_scaled.head()

In [ ]:
# Divisi?n del dataset
# Como todos los valores son iguales, cualquier divisi?n es v?lida
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Tama?o del conjunto de entrenamiento: {X_train.shape}")
print(f"Tama?o del conjunto de prueba: {X_test.shape}")
print(f"\nDistribuci?n en entrenamiento:")
print(y_train.value_counts().sort_index())
print(f"\nDistribuci?n en prueba:")
print(y_test.value_counts().sort_index())

In [ ]:
# Implementaci?n del Modelo Constante
class ConstantFrequencyModel(BaseEstimator, ClassifierMixin):
    """
    Modelo constante que siempre predice el mismo valor.
    Apropiado cuando todos los valores de la variable objetivo son iguales.
    """
    
    def __init__(self, constant_value=1.0):
        self.constant_value = constant_value
    
    def fit(self, X, y):
        # Verificar que todos los valores son iguales
        unique_values = np.unique(y)
        if len(unique_values) == 1:
            self.constant_value = unique_values[0]
            print(f"Modelo constante ajustado. Valor constante: {self.constant_value}")
        else:
            print(f"Advertencia: Los datos tienen m?ltiples valores ?nicos: {unique_values}")
            self.constant_value = np.mean(y)  # Usar la media como fallback
        
        self.classes_ = unique_values
        return self
    
    def predict(self, X):
        # Predecir el valor constante para todas las muestras
        return np.full(X.shape[0], self.constant_value)
    
    def predict_proba(self, X):
        # Para compatibilidad con m?tricas de clasificaci?n
        # Retorna probabilidad 1.0 para la clase constante
        n_samples = X.shape[0]
        if hasattr(self, 'classes_'):
            n_classes = len(self.classes_)
            proba = np.zeros((n_samples, n_classes))
            # Encontrar el ?ndice de la clase constante
            constant_idx = np.where(self.classes_ == self.constant_value)[0][0]
            proba[:, constant_idx] = 1.0
            return proba
        else:
            return np.ones((n_samples, 1))
    
    def score(self, X, y):
        # Calcular accuracy
        predictions = self.predict(X)
        return accuracy_score(y, predictions)

print("Modelo ConstantFrequencyModel implementado")

In [ ]:
# Entrenamiento del modelo constante
constant_model = ConstantFrequencyModel()
constant_model.fit(X_train, y_train)

# Realizar predicciones
y_train_pred = constant_model.predict(X_train)
y_test_pred = constant_model.predict(X_test)

print("Modelo constante entrenado y predicciones generadas")
print(f"Valor constante predicho: {constant_model.constant_value}")
print(f"Predicciones de entrenamiento (muestra): {y_train_pred[:5]}")
print(f"Predicciones de prueba (muestra): {y_test_pred[:5]}")

In [ ]:
# Evaluaci?n del modelo constante
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# M?tricas de entrenamiento
train_mae = mean_absolute_error(y_train, y_train_pred)
train_accuracy = accuracy_score(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# M?tricas de prueba
test_mae = mean_absolute_error(y_test, y_test_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("=== M?TRICAS DEL MODELO CONSTANTE ===")
print(f"\nEntrenamiento:")
print(f"  MAE: {train_mae:.6f}")
print(f"  Accuracy: {train_accuracy:.6f}")
print(f"  R?: {train_r2:.6f}")

print(f"\nPrueba:")
print(f"  MAE: {test_mae:.6f}")
print(f"  Accuracy: {test_accuracy:.6f}")
print(f"  R?: {test_r2:.6f}")

print(f"\n=== INTERPRETACI?N ===")
print(f"Al tener todos los valores iguales a 1.0:")
print(f"- MAE = 0.0 (predicci?n perfecta)")
print(f"- Accuracy = 1.0 (100% de acierto)")
print(f"- R? puede ser indefinido debido a varianza cero")

In [ ]:
# C?lculo del coeficiente de Gini
def calculate_gini_coefficient(y_true, y_pred):
    """
    Calcula el coeficiente de Gini para medir la capacidad discriminatoria del modelo.
    Para un modelo constante con una sola clase, el Gini ser? 0.
    """
    # Verificar si hay variabilidad en las predicciones
    if len(np.unique(y_pred)) == 1:
        print("Advertencia: Predicciones constantes. Gini = 0")
        return 0.0
    
    # Si hay variabilidad, calcular Gini normalmente
    from sklearn.metrics import roc_auc_score
    try:
        # Para clasificaci?n binaria/multinomial
        if len(np.unique(y_true)) > 1:
            auc = roc_auc_score(y_true, y_pred, multi_class='ovr')
            gini = 2 * auc - 1
        else:
            gini = 0.0
    except:
        gini = 0.0
    
    return gini

# Calcular Gini para el modelo constante
train_gini = calculate_gini_coefficient(y_train, y_train_pred)
test_gini = calculate_gini_coefficient(y_test, y_test_pred)

print(f"=== COEFICIENTE DE GINI ===")
print(f"Entrenamiento: {train_gini:.6f}")
print(f"Prueba: {test_gini:.6f}")
print(f"\nInterpretaci?n: Gini = 0 indica que el modelo no discrimina")
print(f"(esperado para modelo constante con una sola clase)")

In [ ]:
# Guardar el modelo entrenado
import os

# Crear directorio models si no existe
os.makedirs('../models', exist_ok=True)

# Guardar modelo constante
model_filename = '../models/frecuencia_rc_constant_model.pkl'
joblib.dump(constant_model, model_filename)

print(f"=== MODELO GUARDADO ===")
print(f"Archivo: {model_filename}")
print(f"Tipo de modelo: ConstantFrequencyModel")
print(f"Valor constante: {constant_model.constant_value}")

# Verificar que el archivo se guard? correctamente
if os.path.exists(model_filename):
    file_size = os.path.getsize(model_filename)
    print(f"Tama?o del archivo: {file_size} bytes")
    print("? Modelo guardado exitosamente")
else:
    print("? Error al guardar el modelo")

In [ ]:
# Prueba de carga del modelo guardado
try:
    # Cargar modelo
    loaded_model = joblib.load(model_filename)
    
    # Verificar que funciona correctamente
    test_predictions = loaded_model.predict(X_test[:5])
    
    print(f"=== VERIFICACI?N DEL MODELO CARGADO ===")
    print(f"Modelo cargado exitosamente: {type(loaded_model).__name__}")
    print(f"Valor constante: {loaded_model.constant_value}")
    print(f"Predicciones de prueba: {test_predictions}")
    print(f"? Modelo funciona correctamente despu?s de la carga")
    
except Exception as e:
    print(f"? Error al cargar el modelo: {e}")

## Resumen y Conclusiones

### Hallazgos Principales

1. **Variable Objetivo**: Todos los 67 registros tienen `Resp_Civil_siniestros_num = 1.0`
2. **Modelo ?ptimo**: Modelo constante que siempre predice 1.0
3. **M?tricas Perfectas**: MAE = 0.0, Accuracy = 100%
4. **Gini = 0**: No hay capacidad discriminatoria (esperado para modelo constante)

### Justificaci?n Actuarial

En el contexto de seguros para estudiantes con cobertura de Responsabilidad Civil:
- La frecuencia constante de 1.0 sugiere que todos los asegurados siniestrados tienen exactamente un siniestro
- Esto puede reflejar el comportamiento espec?fico de esta cobertura en la poblaci?n estudiantil
- El modelo constante es actuarialmente apropiado y computacionalmente eficiente

### Modelo Final

- **Archivo**: `frecuencia_rc_constant_model.pkl`
- **Predicci?n**: Siempre 1.0 para cualquier asegurado siniestrado
- **Uso**: Para el proceso de tarificaci?n de la cobertura RC